# Music Recommendor Algorithm
### Collaborators: Aisha Mardini, Lilly Peters, Shinung Li

In [ ]:
def decode_songs(song_indices, song_encoder):
    """
    Function that decodes the label encoded songs

    ARGS:
    song_indecies: the label encoded songs
    spng_encoder: label encoding 

    RETURNS:
    the decoded songs
    """
    song_indices = np.atleast_1d(song_indices)
    
    return song_encoder.inverse_transform(song_indices)

In [ ]:
def at_k(W, H, train_matrix, test_lookup, k=10):
    """
    https://www.evidentlyai.com/ranking-metrics/precision-recall-at-k
    Function to return the Precision@K which is how relevant the top K recommended songs are and Recall@K which is how many of the 
    recommended songs are relevant are in the top K  

    ARGS:
    W: user latent features
    H: song latent features
    train_matrix: sparse matrix
    test_lookup: dictionary of grouped testing data to compare for relevance
    k: top K songs that we compare

    RETURNS:
    Precision@K, Recall@K value
    """

    precisions = []
    recall = []

    # looks at the relevance for each user then averages it out
    for user_idx in test_lookup.keys():

        # the top k recommendations for each user
        recommended = recommend_user(user_idx,W,H,train_matrix,top_k=k)
        
        # all the songs that a user listened to
        listened = test_lookup[user_idx]

        # relevance is def as overlap between how many are listened to from the top k recommendations
        relevant = len(set(recommended) & listened)
        
        # precision = number of relevant items in K / total number of items in K
        precisions.append(relevant / k)

        # recall = number of relevant items in K / total number of relevant items
        if len(listened) > 0:
            recall.append(relevant / len(listened))

    return np.mean(precisions), np.mean(recall)

In [ ]:
def user_at_k(user_idx, W, H, train_matrix, test_lookup, k=10):

    recommended = recommend_user(user_idx,W,H,train_matrix,top_k=k)

    listened = test_lookup.get(user_idx, set())

    relevant = len(set(recommended) & listened)

    precision = relevant / k

    if len(listened) > 0:
        recall = relevant / len(listened)
    else:
        recall = 0

    return precision, recall

In [ ]:
def recommend_user(user_idx, W, H, train_matrix, top_k=10):
    """
    Function to return K top recommended songs based on a user

    ARGS:
    user_idx: the user to recommend to
    W: user latent features
    H: song latent features
    train_matrix: the sparse matrix
    top_K: integer value of TOP K songs to return (default 10)

    RETURNS:
    top_song_indices: sorted songs by top K recommended
    """

    # predict scores based on user id
    scores = W[user_idx] @ H

    # REMOVE already seen songs
    seen_songs = train_matrix[user_idx].indices
    scores[seen_songs] = -np.inf

    # sort songs by most recommended
    top_song_indices = np.argsort(scores)[::-1][:top_k]

    return top_song_indices

In [ ]:
def train_test_split(df, test_size=0.4):
    """
    Function to randomly shuffle our data frame by each user  
    Split it into training and testing dataframes

    ARGS:
        df - user dataframe
        test_size - default 0.2. 20% of each user into testing 

    RETURNS: split dataframes
        train_df: the training dataframe (1-test_size) %
        test_df: testing dataframe  (test_size) %
    """

    train_rows = []
    test_rows = []

    # splitting based on user so that there is TEST_SIZE % in testing of each user
    for user, user_df in df.groupby('user_idx'):

        # shuffle user df so that we can have random train/testing 
        user_df = user_df.sample(frac=1, random_state=42)

        n_test = max(1, int(len(user_df) * test_size))
        test = user_df.iloc[:n_test]
        train = user_df.iloc[n_test:]

        train_rows.append(train)
        test_rows.append(test)

    # make into pandas df
    train_df = pd.concat(train_rows)
    test_df = pd.concat(test_rows)

    return train_df, test_df

## Step 1: Upload and preprocess datasets

## First dataset:
- https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset
- Used for content based filtering

\
___Elements___

- **track_id:** The Spotify ID for the song
- **artists:** The artists' names. If there is more than one artist, they are separated by a ;
- **album_name:** The album name
- **track_name:** Name of the track
- **popularity:** The popularity is calculated by Spotify's algorithm from [0,100]. Based on the total number of plays the track has had and how recent those plays are. 
- **duration_ms:** The song length in milliseconds
- **explicit:** Whether or not the song has explicit lyrics (true = yes; false = no OR unknown)
- **danceability:** How suitable a track is for dancing from [0,1] based on a combination of elements including tempo, rhythm stability, beat strength, and overall regularity. 
- **energy:** Represents a perceptual measure of intensity and activity from [0,1]. Typically, energetic tracks feel fast, loud, and noisy. 
- **key:** The key the track is in mapped using standard Pitch Class notation. E.g. 0 = C, 1 = C♯/D♭, 2 = D, and so on. If no key was detected, the value is -1
- **loudness:** How loud the track is in decibels
- **mode:** Mode indicates the modality (major or minor) of a trackMajor is represented by 1 and minor is 0
- **speechiness:** Speechiness detects the presence of spoken words in a track from [0,1]
- **acousticness:** Whether the track is acoustic from [0,1]. 1.0 represents high confidence the track is acoustic
- **instrumentalness:** Predicts whether a track contains no vocals from [0,1]
- **liveness:** Detects the presence of an audience in the recording from [0,1]
- **valence:** Musical positiveness conveyed by a track from [0,1]. Tracks with high valence sound more positive (e.g. happy, cheerful, euphoric), while tracks with low valence sound more negative (e.g. sad, depressed, angry)
- **tempo:** Estimated tempo of a track in beats per minute (BPM). 
- **time_signature:** An estimated time signature. The time signature (meter) is a notational convention to specify how many beats are in each bar (or measure). The time signature ranges from 3 to 7 indicating time signatures of 3/4, to 7/4.
- **track_genre:** The genre of the song

In [ ]:
# import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import NMF
import random

In [ ]:
# Loading in Spotify datasets
df_read = pd.read_csv('/Users/aishamardini/Downloads/PIC16B/dataset.csv') # MUST DOWNLOAD LOCALLY AND CHANGE PATH
df_read.head()

In [ ]:
# checking if there are NaN values in track id, artists, albums
df_read[df_read["track_id"].isna()]

In [ ]:
df_read[df_read["artists"].isna()]

In [ ]:
df_read[df_read["album_name"].isna()]

In [ ]:
# drop nan values
df_read = df_read.dropna()

In [ ]:
# check to see if any rows are duplicates and drop them
#print(df_read.drop_duplicates().any())
df_read = df_read.drop_duplicates()

In [ ]:
# checking the types for columns
df_read.dtypes

In [ ]:
# drop the unnamed column
df = df_read.drop(columns=['Unnamed: 0'])

In [ ]:
# check column names
print("Columns: ", df.columns)

In [ ]:
# check statistics
df.describe()

In [ ]:
# check to see if multiartists are separated by semicolon
df[df["artists"].str.contains(";")]["artists"].unique()[:20]

In [ ]:
# how many loudness columns are 0
(df["loudness"] == 0).sum()

In [ ]:
# how many instrumentalness columns are 0
(df["instrumentalness"] == 0).sum()

In [ ]:
# how many tempo columns are 0
(df["tempo"] == 0).sum()

In [ ]:
# drop temp columns that are 0
df = df[df["tempo"] != 0]

In [ ]:
# remove artist's duplicated songs
df = df.sort_values("popularity", ascending=False).drop_duplicates(subset=["track_name", "artists"], keep="first")

In [ ]:
# check to see dropped
df.duplicated(subset=["track_name", "artists"]).sum()

In [ ]:
# remove songs with popularity == 0
df = df[df["popularity"] > 0.1]

In [ ]:
# check statistics again
df.describe()

## 2nd Dataset
\
Second Dataset:
- https://www.kaggle.com/competitions/kkbox-music-recommendation-challenge/data
- Download: train.csv, songs.csv, members.csv, and song_extra_info.csv
- Used for collaborative filtering

In [ ]:
# load in all 2nd datasets; view columns
# DOWNLOAD ALL LOCALLY AND CHANGE PATHS
train = pd.read_csv('/Users/aishamardini/Downloads/PIC16B/train.csv')
songs = pd.read_csv('/Users/aishamardini/Downloads/PIC16B/songs.csv')
members = pd.read_csv('/Users/aishamardini/Downloads/PIC16B/members.csv')
song_extra = pd.read_csv('/Users/aishamardini/Downloads/PIC16B/song_extra_info.csv')

In [ ]:
# print info of all datasets
datasets = [train, songs, members, song_extra]
d = ["train", "songs", "members", "song_extra_info"]

for i, ds in enumerate(datasets):
    print(d[i] + ".csv information:")
    print("Column names: ", ds.columns)
    print("Number of duplicates: ", ds.duplicated().sum())
    print("Number of null in columns: \n", ds.isnull().sum())
    print()

In [ ]:
# print data
train

In [ ]:
# 2nd dataset drop NAN
members = members[['bd', 'msno', 'city']]
members.dropna()

In [ ]:
# FIRST MERGE: add a song name column by merging the 2 datasets based on song id

merged_song = pd.merge(songs, song_extra[['song_id', 'name']], on='song_id', how='left')
merged_song = merged_song.drop_duplicates()
merged_song.dropna()

# keep needed columns
merged_song = merged_song[['artist_name', 'name', 'song_id']]

print("1st Merged Dataset (Song Names and ID):")
merged_song

In [ ]:
# SECOND merging 
colab_df = pd.merge(train, merged_song[['song_id', 'name', 'artist_name']], on='song_id', how='left')
colab_df

In [ ]:
# FINAL merged and clean dataset
collab = pd.merge(colab_df, members[['bd', 'msno', 'city']], on='msno', how='left')

# rename columns to be more explicit
collab = collab.rename(columns={'bd': 'age', 'msno':'user', 'target':'repeated'})

# source_system_tab sees where the song was found (either in the playlist, search, discover, explore, radio, or listened with someone)
collab = collab.dropna(subset=['source_system_tab'])

# encode source system tab column 1-6
# strongest signal to weakest
collab['source_system_tab'] = collab['source_system_tab'].map(
    {
    'my library': 6, 
    'search': 5, 
    'discover': 4,      
    'explore': 3, 
    'radio': 2,
    'listen with': 1 
    })


# view unique values
print("Unique outputs in source_system_tab ", collab['source_system_tab'].unique())
print() 
# view distribution of values -- most listened to songs are from the library
print("Count of each source system tab ", collab['source_system_tab'].value_counts())

print()
# print column names for final dataset
print("Columns: ", collab.columns)

print()
print("FINAL Merged Dataset:")
collab

In [ ]:
# cleaning data further

collab = collab[(collab['age'] > 0) & (collab['age'] < 100)] # remove all instances where age is 0 or impossible

collab.isnull().sum() # count number of NAs
collab = collab.dropna()   # drop Nas
collab.isnull().sum()    # check to make sure actually dropped


pd.set_option('display.float_format', '{:.2f}'.format)
collab.info()
collab.isnull().sum()
collab.describe()

In [ ]:
# clean users
usersdropped = 80

print("Number of Unique users ", collab["user"].nunique())  # number of unique users
user_counts = collab["user"].value_counts()

# how many users show up less than usersdropped times
print(f"Users with <{usersdropped} interactions:", (user_counts < usersdropped).sum())

# keep users who appear more than usersdropped times 
collab = collab[collab["user"].isin(user_counts[user_counts > usersdropped].index)]
print()
# clean songs
songsdropped = 50

print("Number of Unique songs " , collab["song_id"].nunique())  # number of unique song ids ==> songs
songid_counts = collab["song_id"].value_counts()

# how many songs show up less than songsdropped times
print(f"Songs with <{songsdropped} listens:", (songid_counts < songsdropped).sum())

# keep songs that appear more than songsdropped times 
collab = collab[collab["song_id"].isin(songid_counts[songid_counts > songsdropped].index)]

## Step 2: Data Analysis

In [ ]:
# understanding member stats

collab.groupby('city')['song_id'].value_counts()

city_songs = collab.groupby(['city', 'song_id'])['repeated'].sum().reset_index()
city_songs = city_songs.sort_values(['city', 'repeated'], ascending=[True, False])

# see top 5 songs per city
city_songs.groupby('city').head(5)

collab.groupby('city')['song_id'].nunique().plot(kind='bar')
plt.xlabel('city')
plt.ylabel('unique songs')
plt.savefig('city.png', dpi=300, bbox_inches='tight')

collab.boxplot(column='age', by='city')

plt.show()

In [ ]:
# City 13 had the most replays and the most unique songs. 
# City 5 had the second most unique songs, and city 15 and city 22 had a 
# close number of unique songs (count). 
collab.groupby('city')['repeated'].sum().plot(kind='bar')
plt.ylabel('total replays')
plt.xlabel('city')
plt.show()

In [ ]:
#age distribition

collab['age'].plot(kind='hist', bins=30) # visualize age distribution counts
plt.xlim(0, 80) # shorten the x axis so it's a clearer picture
plt.show()

# correlation heatmap
sns.heatmap(collab.corr(numeric_only=True), annot=True, fmt=".1f")
plt.show()

#frequency table for source system tabs
collab['source_system_tab'].plot(kind='hist', bins=10)
plt.xlim(0, 10)
plt.show()

In [ ]:
# bar plot of the first 15 most popular genres
plt.title("Top 15 Popular Genres")
genre_pop = df.groupby("track_genre")["popularity"].mean().sort_values(ascending=False).head(15)
sns.barplot(x=genre_pop.index, y=genre_pop.values)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# plot for the 15 least popular genres
plt.title("15 Least Popular Genres")
genre_pop = df.groupby("track_genre")["popularity"].mean().sort_values(ascending=True).head(15)
sns.barplot(x=genre_pop.index, y=genre_pop.values)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# heatmap plot correlating features
plt.title("Correlation Heatmap")
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".1f")
plt.show()

## Step 3: Collaborative Filtering

In [ ]:
# drop unnecessary columns
collab = collab[['user', 'song_id', 'name', 'repeated']]
collab

In [ ]:
# Unique count
print("Number of Unique users ", collab["user"].nunique())
print("Number of Unique songs ", collab["song_id"].nunique())

In [ ]:
# create matrix for Nonnegative matrix factorization
# rows are user, columns are song, filled with 1 if they repeated a song and 0 otherwise

# label encode the users and songs to build the sparse matrix
user_encoder = LabelEncoder()
song_encoder = LabelEncoder()

collab['user_idx'] = user_encoder.fit_transform(collab['user'])
collab['name_idx'] = song_encoder.fit_transform(collab['name'])

train_df, test_df = train_test_split(collab)

# build the matrix
matrix = csr_matrix((train_df['repeated'], (train_df['user_idx'], train_df['name_idx'])))

# check the sizes + should match the matrix size
print("Amt of users:", len(user_encoder.classes_))
print("Amt of songs:", len(song_encoder.classes_))
print("Matrix size:", matrix.shape)

In [ ]:
# Why we use NMF: 
# https://communities.sas.com/t5/SAS-Communities-Library/Non-negative-Matrix-Factorization-Part-1-Understanding-Its/ta-p/973592

# NMF model
nmf_model = NMF(n_components=30, init='nndsvd', random_state=42, max_iter=900)

# lower rank decomposed matricies
W = nmf_model.fit_transform(matrix) # user latent features
H = nmf_model.components_ # song latent features / weights
print(W.shape, H.shape)

In [ ]:
# print out top 10 songs from each factor to interpret what the machine learned
for factor in range(W.shape[1]):

    topSongs = np.argsort(H[factor])[::-1][:10]

    print(f"\nFactor {factor}")

    print(song_encoder.inverse_transform(topSongs))


In [ ]:
# print all users top factor and the array of song latent feature weights
dominantFactor = np.argmax(W, axis=1)

for i in range(W.shape[0]):
    print(f"User {i}: Factor {dominantFactor[i]}")
    print(f"User {i}'s latent features:", W[i][:])
    print()

In [ ]:
usercount = len(user_encoder.classes_)
# list 10 random users
users = random.sample(range(usercount), 10)

for user in users:
    recommended_indices = recommend_user(user,W,H, matrix,top_k=10)
    recommended_songs = decode_songs(recommended_indices,song_encoder)
    print(f"User #{user} - Dominant factor: {dominantFactor[user]}")
    print(recommended_songs)
    print()

In [ ]:
# print all song's top 3 factors
for song_idx in range(H.shape[1]):

    song_name = decode_songs(song_idx, song_encoder)[0]

    top_factors = np.argsort(H[:, song_idx])[::-1][:3]

    print(f"\n{song_name}")

    for factor in top_factors:
        print(f"Factor {factor}: "f"{H[factor, song_idx]:.3f}")

In [ ]:
# dictionary of user test set with songs they listened to but the machine hasnt seen
test_lookup = (test_df.groupby('user_idx')['name_idx'].apply(set).to_dict())
test_lookup

In [ ]:
top_k = [2, 4, 6, 8, 10, 12]
precision_scores = []
recall_scores = []

# see the relationship between precision and recall at different k values
for k in top_k:
    
    precision, recall = at_k(W,H,matrix,test_lookup,k=k)
    print(f"Precision@{k}:", precision)
    print(f"Recall@{k}:", recall)
    print()
    precision_scores.append(precision)
    recall_scores.append(recall)

In [ ]:
# plotting precision vs recall @ different k values
fig, ax = plt.subplots(1,2,figsize=(10,5))


ax[0].plot(top_k, precision_scores, label = f"Precision@k = {k}")
ax[1].plot(top_k, recall_scores, label = f"Recall@k = {k}")

ax[0].set_xlabel("k")
ax[0].set_ylabel("Precision")
ax[0].set_title("Precision@K")
ax[1].set_xlabel("k")
ax[1].set_ylabel("Recall")
ax[1].set_title("Recall@K")

plt.show()

In [ ]:
# precision and recall
k = 10
# amount of users to randomly generate
randusers = 100

usercount = len(user_encoder.classes_)
users = random.sample(range(usercount), randusers)
precision_avg = []
recall_avg = []

for user in users:

    # recommended songs for each user
    recommended_indices = recommend_user(user,W,H,matrix,top_k=k)
    # decode the songs
    recommended_songs = decode_songs(recommended_indices,song_encoder)
    # calcuate precision and recall
    precision, recall = user_at_k(user,W,H,matrix,test_lookup,k=k)

    #  dominant latent factor for each user
    factor = dominantFactor[user]
    top_factor_songs = np.argsort(H[factor])[::-1][:10]

    print("-----------------")
    print(f"User #{user}")
    print(f"Dominant Factor: {factor}")
    print(f"Precision@10: {precision:.3f}")
    print(f"Recall@10: {recall:.3f}")

    precision_avg.append(precision)
    recall_avg.append(recall)

    # print out top k recommended songs
    print("\nTop Recommendations:")
    print(recommended_songs)

    # print out factor songs to compare
    print("\nFactor Theme:")
    print(song_encoder.inverse_transform(top_factor_songs))
    print()

# average out the precision and recall
print(f"{randusers} random users average precision@{k} and recall@{k}: ", np.mean(precision_avg), " and " , np.mean(recall_avg))

In [ ]:
# look at a random users recommendations
user = 4862

recommended = set(recommend_user(user, W, H, matrix, top_k=10))
listened = test_lookup[user]

print("User: ", user)
print()
print("Recommended Songs")
print(decode_songs(list(recommended), song_encoder))

print()
print("Test Songs")
print(decode_songs(list(listened), song_encoder))

print()
print("Overlapping Set")
print(recommended & listened)

## Step 4: Content Based Filtering